In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_lineage_analysis/workflow_metadata"
tgt_silver_table = "data_governance.silver_lineage_analysis.lineage_workflow_metadata"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df.limit(20).display()

In [0]:
df = df.withColumn(
    "job_category",
    when(col("name").contains("DATA QUALITY"), "DATA_QUALITY")
    .when(col("name").contains("DLT"), "DELTA_LIVE_TABLE")
    .when(col("name").contains("CUSTOMER"), "CUSTOMER_PIPELINE")
    .otherwise("OTHER")
)


df = df.withColumn(
    "is_deleted",
    col("delete_time").isNotNull()
)

df = df.withColumn(
    "is_active",
    (~col("paused")) & col("delete_time").isNull()
)

df = df.withColumn("trigger_file_arrival", col("trigger.file_arrival")) \
       .withColumn("trigger_periodic", col("trigger.periodic")) \
       .withColumn("trigger_continuous", col("trigger.continuous")) \
       .withColumn("trigger_table_update", col("trigger.table_update")) \
       .withColumn("trigger_schedule", col("trigger.schedule"))

df = df.drop("trigger")

In [0]:

df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("workspace_id") \
 .saveAsTable(tgt_silver_table)